# SEA-Spoof: Exploratory Data Analysis (EDA)
## สำรวจข้อมูลก่อนใช้ AASIST / RawNet2

Notebook นี้ตอบว่า **ข้อมูลที่มีในเครื่องคืออะไร มีเสียงแบบไหน สมดุลหรือไม่ และมีข้อจำกัดอะไร** ไม่ใช่การวัดความแม่นยำโมเดล

ขอบเขต 3 ระดับ:
1. **Parquet ทุกภาษาที่ดาวน์โหลดไว้** — อ่านทุกแถวของ metadata โดยไม่อ่านคอลัมน์ audio bytes; ไม่อ้างว่าเป็นทั้ง release ถ้ายังไม่มี Train
2. **เสียงไทยที่สกัดแล้ว** — ตรวจ manifest และ header ของไฟล์จริงทุกไฟล์ รวมถึง SHA-256 ของไฟล์เมื่อเปิดตัวเลือก
3. **สัญญาณเสียง** — ค่าเริ่มต้นสุ่มแบบแบ่งกลุ่ม split × label พร้อม seed; อ่านทั้งคลิปของตัวอย่างที่เลือก ไม่ใช่สำรวจ waveform ทุกคลิป เว้นแต่เปลี่ยน SIGNAL_SCOPE เป็น "all"

**ไม่มีการแก้ Parquet / manifest / audio ต้นฉบับ ไม่มีการดาวน์โหลด ไม่มีการฝึก และไม่มีการเลือก threshold จาก Test**
ผล EDA ใช้ทำความเข้าใจและตรวจคุณภาพข้อมูล ไม่ใช้เลือกโมเดลจากผล Test

แหล่งคำอธิบายฟิลด์: [SEA-Spoof dataset card](https://huggingface.co/datasets/Jack-ppkdczgx/SEA-Spoof), ตรวจเอกสารประกอบการสร้าง notebook วันที่ 2 ตุลาคม 2026
ข้อมูลจำกัดเฉพาะผู้ได้รับอนุญาต ผลละเอียดอยู่ใน outputs/ ที่ถูก gitignore; อย่า commit notebook ที่มี transcript/audio ฝังอยู่ หรือแจกจ่ายผลที่มีข้อมูลรายแถวโดยไม่ตรวจสิทธิ์


## 0. วิธีรันและตัวเลือก

ใน VS Code เลือก kernel เป็น .venv ของโปรเจกต์ แล้ว Run All จากบนลงล่าง
ติดตั้งครั้งแรกจาก project root: `\.venv\Scripts\python.exe -m pip install -e ".[eda]"`

- ทุกครั้งอ่านข้อมูลใหม่ ไม่ใช้ cache ที่อาจล้าสมัย
- HASH_AUDIO=True ตรวจไฟล์ไบต์เหมือนกันทุกไฟล์ไทย; ไม่เท่ากับตรวจผู้พูดหรือเสียงที่เข้ารหัสต่างกัน
- SIGNAL_SCOPE="sample" ประหยัดเวลา; "all" ถอดรหัสเสียงไทยทุกคลิปและอาจใช้เวลาหลายนาที
- PLAY_AUDIO=False ป้องกันการฝังเสียงในผล notebook โดยอัตโนมัติ
- SHOW_RAW_EXAMPLES=False ไม่แสดง transcript/row IDs รายตัวในผลปกติ
- กราฟใช้ภาษาอังกฤษเพื่อลดปัญหาฟอนต์ไทย; คำอธิบายอ่านเป็นภาษาไทยได้


In [ ]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from collections import Counter
import hashlib
import importlib.metadata
import itertools
import json
import platform
import sys
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
import soundfile as sf
from scipy import signal as scipy_signal
from IPython.display import display, Markdown, Audio

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").is_file() and (p / "data/manifests").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the project folder.")
EDA_DIR = ROOT / "data/exploration/sea_spoof"
OUTPUT = EDA_DIR / "outputs"
OUTPUT.mkdir(parents=True, exist_ok=True)
SOURCE_ROOT = ROOT / "data/raw/sea_spoof_hf/data"
MANIFEST_NAMES = {"train": "thai_train.csv", "validation": "thai_dev.csv",
                  "evaluation": "thai_test.csv"}
SEED = 20261002
WORKERS = 4
HASH_AUDIO = True
SIGNAL_SCOPE = "sample"  # change to "all" to decode every Thai clip
SAMPLES_PER_SPLIT_LABEL = 60
NEAR_ZERO_THRESHOLD = 1e-4
PLAY_AUDIO = False
SHOW_RAW_EXAMPLES = False
TARGET_SAMPLE_RATE = 16_000
TARGET_SAMPLES = 64_600
WINDOW_S = TARGET_SAMPLES / TARGET_SAMPLE_RATE
STARTED_AT = datetime.now(timezone.utc).isoformat()
assert SIGNAL_SCOPE in {"sample", "all"}
assert SAMPLES_PER_SPLIT_LABEL > 0 and WORKERS > 0
sys.path.insert(0, str(EDA_DIR))
from sea_spoof_eda_utils import (normalized_text, inspect_audio_header,
    inspect_signal, input_window_summary, resolve_audio_path, file_sha256)

sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams.update({"figure.figsize": (10, 4), "figure.dpi": 110})
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 40)
TABLES = {}
def show_table(name, frame, limit=40):
    TABLES[name] = frame.copy()
    display(frame.head(limit))
    if len(frame) > limit:
        print(f"Showing {limit}/{len(frame)} rows; full table will be exported locally.")
def save_plot(name):
    plt.tight_layout()
    plt.savefig(OUTPUT / (name + ".png"), dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
def values(frame, column):
    return frame[column].astype("string").fillna("").str.strip()
def category_values(frame, column):
    return values(frame, column).replace("", "(missing)")
def grouped_counts(frame, columns):
    return frame.groupby(columns, dropna=False).size().rename("clips").reset_index()

versions = {name: importlib.metadata.version(name) for name in
            ["numpy", "pandas", "pyarrow", "soundfile", "matplotlib", "seaborn", "nbformat"]}
display({"python": platform.python_version(), "packages": versions,
         "signal_scope": SIGNAL_SCOPE, "hash_all_thai_audio": HASH_AUDIO,
         "seed": SEED, "input_window_seconds": WINDOW_S})


## 1. Inventory และหลักฐานเวอร์ชันที่ดาวน์โหลด

อ่าน schema / จำนวนแถว / row groups / ขนาดไฟล์จาก Parquet footer ก่อน
อ่าน commit/etag จาก HF download metadata ถ้ามี ไม่เรียก API และไม่ใช้ token
ค่า "unknown" หมายถึงยืนยัน revision จาก cache ไม่ได้; timestamp/ขนาดไฟล์ไม่ใช่หลักฐานความเหมือนของเนื้อหา
ไม่ hash Parquet ขนาดหลาย GB โดยอัตโนมัติ แต่บันทึก fingerprint ของ manifest และ hash เสียงไทยถ้าเปิดตัวเลือก


In [ ]:
shards = sorted(SOURCE_ROOT.glob("*/*.parquet"))
if not shards:
    raise FileNotFoundError("No local SEA-Spoof Parquet files found.")
inventory_rows, schemas, schema_rows = [], {}, []
for path in shards:
    parquet = pq.ParquetFile(path)
    schemas[path.relative_to(ROOT).as_posix()] = str(parquet.schema_arrow)
    cache = SOURCE_ROOT.parent / ".cache/huggingface/download/data" / path.parent.name / (path.name + ".metadata")
    cache_lines = cache.read_text(encoding="utf-8").splitlines() if cache.is_file() else []
    inventory_rows.append({
        "parquet_file": path.relative_to(ROOT).as_posix(),
        "directory_split": path.parent.name, "rows": parquet.metadata.num_rows,
        "row_groups": parquet.metadata.num_row_groups, "size_GiB": path.stat().st_size / 1024**3,
        "mtime_utc": datetime.fromtimestamp(path.stat().st_mtime, timezone.utc).isoformat(),
        "hf_revision": cache_lines[0] if cache_lines else "unknown",
        "hf_etag": cache_lines[1] if len(cache_lines) > 1 else "unknown"})
    schema_rows.extend({"parquet_file": path.name, "column": field.name, "arrow_type": str(field.type)}
                       for field in parquet.schema_arrow)
inventory = pd.DataFrame(inventory_rows)
schema_table = pd.DataFrame(schema_rows)
show_table("parquet_inventory", inventory)
show_table("schema", schema_table.drop_duplicates(["column", "arrow_type"]).drop(columns="parquet_file"))
available_splits = sorted(inventory.directory_split.unique().tolist())
display(Markdown(f"**ขอบเขตจริง:** พบ {len(shards)} shards, split={available_splits}. "
                 f"ยังไม่พบ {sorted(set(MANIFEST_NAMES) - set(available_splits)) or 'ไม่มี split ที่ขาด'} "
                 "ใน Parquet ที่ดาวน์โหลดไว้ — ไม่ใช้ตัวเลขนี้แทนทั้ง release."))


## 2. Data dictionary: แต่ละคอลัมน์หมายถึงอะไร

หนึ่งแถวคือ **หนึ่งคลิป / utterance** ไม่ใช่หนึ่งคน
row_id เป็น key หลักของ release; utterance_id อาจซ้ำข้ามแหล่งได้
source_model เป็นระบบสร้างเสียง ไม่ใช่ AASIST/RawNet2 ที่เราจะใช้ตรวจ
online/offline เป็นหมวดแหล่งสร้างตาม dataset ไม่ใช่เสียงที่เรารับผ่านโทรศัพท์จริง
speaker_or_voice อาจเป็นชื่อ voice ของระบบสังเคราะห์ จึงไม่เท่ากับจำนวนบุคคล
condition="clean" ใน manifest หมายถึงเรายังไม่ได้เพิ่มความเสื่อม ไม่รับรองว่าเสียงต้นฉบับไม่มี noise


In [ ]:
field_notes = {
 "row_id": "รหัสแถวที่ควรไม่ซ้ำทั่ว release; ใช้เชื่อม source กับ manifest",
 "utterance_id": "รหัส utterance ต้นทาง; ไม่รับรอง unique ข้าม source subsets",
 "audio": "struct: FLAC bytes และ logical path; ไม่อ่าน bytes ใน metadata scan",
 "text": "ข้อความที่เชื่อมกับคลิป; ต้องตรวจ provenance / exactness",
 "language": "tag ภาษา; th ไม่ได้ยืนยันภูมิลำเนา/สำเนียง",
 "label": "bonafide=เสียงจริง; spoof=เสียงสังเคราะห์ตาม label ต้นทาง",
 "spoof_type": "ชนิดกลุ่ม spoof หรือ bonafide ตามต้นทาง",
 "category": "หมวดแหล่งเสียง เช่น bonafide/offline_spoof/online_spoof",
 "split": "train/validation/evaluation ทางการ; ไม่สุ่มแบ่งใหม่",
 "text_source": "รายละเอียดแหล่ง transcript; ไม่ถือว่าเป็นการถอดเสียงตรวจแล้วทั้งหมด",
 "mapping_source": "local หรือ whisperx_backfill ตาม provenance",
 "text_granularity": "ระดับการเชื่อมข้อความ เช่น utterance หรือ source/video",
 "is_text_exact": "flag ที่ต้นทางคาดว่าข้อความตรงระดับ utterance; ไม่ใช่การตรวจฟังของเรา",
 "source_model": "ระบบสร้างเสียง spoof; ไม่ใช่ detector ของเรา",
 "source_dataset": "ชุดต้นทางของเสียง; ค่าว่างอาจไม่ applicable สำหรับบาง label",
 "speaker_or_voice": "รหัสผู้พูดหรือ synthetic voice ตามต้นทาง; ห้ามนับเป็นบุคคลโดยอัตโนมัติ",
 "sampling_rate": "sample rate ใน metadata; เทียบกับ header ไฟล์จริง",
 "audio_was_resampled": "flag ว่าต้นทาง resample ตอน packaging หรือไม่",
 "path": "path เสียงไทยที่สกัดในโปรเจกต์",
 "source_audio_path": "logical path ใน source audio struct; ไม่ใช่ local path ของเรา",
 "condition": "clean=ไฟล์ก่อน augmentation ของเรา",
 "source_parquet": "ชื่อ shard ที่ใช้สกัด; ต้องพิจารณาคู่กับ split",
}
dictionary = pd.DataFrame([{"column": name, "meaning": meaning} for name, meaning in field_notes.items()])
show_table("data_dictionary", dictionary)


## 3. อ่าน metadata ทุกแถวของ shards ในเครื่อง (ทุกภาษา)

เลือกคอลัมน์ทั้งหมด **ยกเว้น audio** เพื่อลด RAM และไม่ถอดรหัสเสียงต่างภาษา
directory split และค่าคอลัมน์ split ถูกเก็บแยกไว้ตรวจความสอดคล้อง
ค่าสถิติที่ได้คือข้อมูลใน shards ที่มีจริง ไม่ใช่สถิติ Train ที่ยังไม่ได้โหลด


In [ ]:
metadata_parts = []
for path in shards:
    parquet = pq.ParquetFile(path)
    columns = [name for name in parquet.schema_arrow.names if name != "audio"]
    part = parquet.read(columns=columns).to_pandas()
    part["parquet_file"] = path.name
    part["directory_split"] = path.parent.name
    metadata_parts.append(part)
source = pd.concat(metadata_parts, ignore_index=True)
required = {"row_id", "language", "label", "split", "sampling_rate"}
if not required.issubset(source.columns):
    raise ValueError(f"Required source columns missing: {required - set(source.columns)}")
source_counts = grouped_counts(source, ["directory_split", "language", "label"])
show_table("source_counts_all_languages", source_counts)
show_table("source_split_language", pd.crosstab(source.directory_split, source.language).reset_index())
show_table("source_language_label", pd.crosstab(source.language, source.label).reset_index())
print(f"Read {len(source):,} metadata rows; memory ≈ {source.memory_usage(deep=True).sum()/1024**2:.1f} MiB.")
language_plot = pd.crosstab(source.language, source.label)
language_plot.plot.bar(stacked=True)
plt.title("Available local Parquet: clips by language and label")
plt.ylabel("Clips"); plt.xlabel("Language tag")
save_plot("01_all_languages_labels")
source_integrity = pd.DataFrame([
 {"check": "blank row_id", "affected_rows": int(values(source, "row_id").eq("").sum())},
 {"check": "duplicate row_id (all occurrences)", "affected_rows": int(source.row_id.duplicated(keep=False).sum())},
 {"check": "invalid label", "affected_rows": int((~source.label.isin(["bonafide", "spoof"])).sum())},
 {"check": "split disagrees with directory", "affected_rows": int((source.split != source.directory_split).sum())},
 {"check": "metadata sample rate not 16000", "affected_rows": int((pd.to_numeric(source.sampling_rate, errors="coerce") != 16000).sum())},
])
show_table("source_integrity", source_integrity)


## 4. โหลด Thai manifests และตรวจเทียบกับ source

Manifest เป็นรายการ path + label + metadata ที่สกัด ไม่ใช่ตัวเสียงเอง
ตรวจว่าทุก row_id ไทยใน local Parquet มีใน manifest และ metadata ตรงกัน
ถ้า Train เพิ่มในอนาคต notebook จะอ่านโดยอัตโนมัติจากชื่อ thai_train.csv
ไม่พิมพ์ transcript หรือรหัสรายแถวในค่าตั้งปกติ


In [ ]:
manifest_parts, manifest_inventory = [], []
for split, name in MANIFEST_NAMES.items():
    path = ROOT / "data/manifests" / name
    if not path.is_file():
        continue
    part = pd.read_csv(path, dtype="string", keep_default_na=False)
    part["manifest_split"] = split
    part["manifest_name"] = name
    manifest_parts.append(part)
    manifest_inventory.append({"manifest": name, "split": split, "rows": len(part),
                               "sha256": file_sha256(path)})
if not manifest_parts:
    raise FileNotFoundError("No Thai manifests; run scripts/extract_thai_parquet.py first.")
thai = pd.concat(manifest_parts, ignore_index=True)
if not {"path", "row_id", "label", "language", "split"}.issubset(thai.columns):
    raise ValueError("Thai manifest has missing required fields.")
show_table("manifest_inventory", pd.DataFrame(manifest_inventory))
if SHOW_RAW_EXAMPLES:
    display(thai.head(5))
else:
    print("Raw transcript/ID examples hidden. Set SHOW_RAW_EXAMPLES=True only for local inspection.")
manifest_integrity = pd.DataFrame([
 {"check": "blank row_id", "affected_rows": int(values(thai, "row_id").eq("").sum())},
 {"check": "duplicate row_id (all occurrences)", "affected_rows": int(thai.row_id.duplicated(keep=False).sum())},
 {"check": "duplicate local audio path", "affected_rows": int(thai.path.duplicated(keep=False).sum())},
 {"check": "non-th language tag", "affected_rows": int(thai.language.ne("th").sum())},
 {"check": "invalid label", "affected_rows": int((~thai.label.isin(["bonafide", "spoof"])).sum())},
 {"check": "split disagrees with manifest name", "affected_rows": int(thai.split.ne(thai.manifest_split).sum())},
])
show_table("manifest_integrity", manifest_integrity)
source_thai = source.loc[source.language.eq("th")].copy()
reconciliation = []
common_columns = [c for c in source_thai.columns if c in thai.columns and c != "row_id"]
for split in sorted(set(source_thai.directory_split) | set(thai.manifest_split)):
    left = source_thai.loc[source_thai.directory_split.eq(split)]
    right = thai.loc[thai.manifest_split.eq(split)]
    source_ids, manifest_ids = set(left.row_id), set(right.row_id)
    record = {"split": split, "source_thai_rows": len(left), "manifest_rows": len(right),
              "source_ids_missing_in_manifest": len(source_ids - manifest_ids),
              "manifest_ids_not_in_source": len(manifest_ids - source_ids)}
    reconciliation.append(record)
show_table("source_manifest_coverage", pd.DataFrame(reconciliation))
if thai.row_id.duplicated().any() or source_thai.row_id.duplicated().any():
    raise ValueError("Duplicate row_id found: inspect integrity table before one-to-one audio/source joins.")
paired = source_thai.merge(thai, on="row_id", suffixes=("_source", "_manifest"), validate="one_to_one")
metadata_mismatches = []
for column in common_columns:
    left = values(paired, column + "_source")
    right = values(paired, column + "_manifest")
    if column in {"is_text_exact", "audio_was_resampled"}:
        left, right = left.str.lower(), right.str.lower()
    metadata_mismatches.append({"column": column, "mismatched_rows": int(left.ne(right).sum())})
show_table("source_manifest_metadata_mismatches", pd.DataFrame(metadata_mismatches))


## 5. คุณภาพ metadata: missing, dtype, unique และค่าที่พบ

ค่าว่างบางคอลัมน์เป็น **not applicable** ไม่ใช่เสียหาย เช่น source_model ของเสียงจริง
จึงแสดง missing แยกตาม label ไม่เติมค่าหรือเดาเอง
จำนวน unique ใน speaker_or_voice เป็นจำนวนค่ารหัสที่พบ ไม่ใช่จำนวนบุคคล
หากไม่มี age/gender/province ไม่สามารถอนุมานประชากรจากเสียงหรือชื่อไฟล์ใน notebook นี้


In [ ]:
profile_rows = []
for column in thai.columns:
    text_values = values(thai, column)
    profile_rows.append({"column": column, "dtype": str(thai[column].dtype),
        "missing_or_blank": int(text_values.eq("").sum()),
        "missing_pct": float(text_values.eq("").mean() * 100),
        "unique_nonblank": int(text_values.loc[text_values.ne("")].nunique())})
profile = pd.DataFrame(profile_rows).sort_values("missing_pct", ascending=False)
show_table("thai_metadata_profile", profile)
missing_by_label = pd.DataFrame({label: pd.Series({
    column: float(values(group, column).eq("").mean() * 100) for column in thai.columns})
    for label, group in thai.groupby("label")}).rename_axis("column").reset_index()
show_table("missing_pct_by_label", missing_by_label)
missing_heatmap = missing_by_label.set_index("column")
plt.figure(figsize=(10, max(6, .38 * len(missing_heatmap))))
sns.heatmap(missing_heatmap, cmap="YlOrRd", vmin=0, vmax=100, annot=True, fmt=".0f", yticklabels=True)
plt.title("Blank metadata (%) by label"); plt.xlabel("Label")
save_plot("02_missing_metadata")
demographic_aliases = {
 "age": {"age", "age_group", "speaker_age"},
 "gender": {"gender", "sex", "speaker_gender"},
 "region_or_accent": {"province", "district", "region", "accent", "accents", "locale", "speaker_region"},
}
demographics = pd.DataFrame([{"dimension": dimension,
    "available_columns": ", ".join(sorted(set(thai.columns) & aliases)) or "(absent)"}
    for dimension, aliases in demographic_aliases.items()])
show_table("demographic_availability", demographics)
speaker_coverage = thai.assign(has_speaker_value=values(thai, "speaker_or_voice").ne("")).groupby(
    ["manifest_split", "label"]).agg(clips=("row_id", "size"),
    nonblank_speaker_or_voice=("has_speaker_value", "sum")).reset_index()
show_table("speaker_or_voice_coverage", speaker_coverage)
display(Markdown("**ข้อจำกัด:** language=th ไม่ได้ยืนยันสำเนียง/ถิ่น "
                 "และ speaker_or_voice ของ spoof อาจเป็น synthetic voice ไม่ใช่มนุษย์ "
                 "แม้ไม่มี row_id ซ้ำก็ยังไม่ยืนยัน speaker-disjoint splits."))


## 6. ความสมดุลและแหล่งที่มา

ดู label × split, source_dataset ของเสียงจริง, source_model ของเสียงสังเคราะห์
และหมวด online/offline ตาม metadata พร้อมสัดส่วน ไม่ใช่ดูจำนวนรวมอย่างเดียว
ความต่างของ source ระหว่าง label อาจทำให้ detector เรียนรู้ลักษณะการบันทึกแทน artefact สังเคราะห์ได้ — เป็นความเสี่ยงที่ต้องทดสอบ ไม่ใช่ข้อสรุปว่าเกิดขึ้นแล้ว


In [ ]:
label_counts = grouped_counts(thai, ["manifest_split", "label"])
label_counts["pct_within_split"] = label_counts.clips / label_counts.groupby("manifest_split").clips.transform("sum") * 100
show_table("thai_label_balance", label_counts)
pd.crosstab(thai.manifest_split, thai.label).plot.bar(stacked=True)
plt.title("Thai clips: official split and label"); plt.ylabel("Clips"); plt.xlabel("Split")
save_plot("03_thai_label_balance")
source_fields = ["category", "spoof_type", "source_dataset", "source_model", "speaker_or_voice",
                 "audio_was_resampled"]
for column in source_fields:
    table = grouped_counts(thai.assign(**{column: category_values(thai, column)}),
                           ["manifest_split", "label", column])
    table["pct_within_split_label"] = table.clips / table.groupby(
        ["manifest_split", "label"]).clips.transform("sum") * 100
    show_table("thai_" + column + "_counts", table, limit=30)
real = thai.loc[thai.label.eq("bonafide")].copy()
fake = thai.loc[thai.label.eq("spoof")].copy()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
category_values(real, "source_dataset").value_counts().plot.barh(ax=axes[0])
category_values(fake, "source_model").value_counts().plot.barh(ax=axes[1])
axes[0].set_title("Bonafide: source datasets"); axes[1].set_title("Spoof: synthesis sources")
for ax in axes: ax.set_xlabel("Clips")
save_plot("04_thai_source_mix")


## 7. Transcript: ความยาวและความน่าเชื่อถือของ mapping

นับตัวอักษร ไม่ใช้ whitespace word count เป็นจำนวนคำไทย เพราะภาษาไทยไม่ได้คั่นทุกคำด้วยช่องว่าง
normalized_text ใช้ NFC + รวมช่องว่างเท่านั้น ไม่ลบวรรณยุกต์หรือเครื่องหมาย
ข้อความซ้ำในคู่จริง/สังเคราะห์อาจเป็นการออกแบบ paired transcripts ไม่ใช่ duplicate audio
ข้อความที่ generated/backfilled หรือระดับ source/video ไม่รับรองว่าเป็น transcript ตรงคลิปทุกตัว


In [ ]:
thai["normalized_text"] = thai.text.map(normalized_text)
thai["text_chars"] = thai.normalized_text.str.len()
thai["has_thai_character"] = thai.normalized_text.str.contains("[\u0E00-\u0E7F]", regex=True)
show_table("text_length_summary", thai.groupby(["manifest_split", "label"]).text_chars.describe(
    percentiles=[.05, .25, .5, .75, .95]).reset_index())
for column in ["mapping_source", "text_source", "text_granularity", "is_text_exact"]:
    show_table("transcript_" + column, grouped_counts(
        thai.assign(**{column: category_values(thai, column)}), ["manifest_split", "label", column]), limit=20)
show_table("text_quality_flags", pd.DataFrame([
 {"check": "blank transcript", "clips": int(thai.text_chars.eq(0).sum())},
 {"check": "no Thai character in nonblank transcript", "clips": int((thai.text_chars.gt(0) & ~thai.has_thai_character).sum())},
 {"check": "duplicate normalized nonblank text (all occurrences)", "clips": int(
     thai.loc[thai.text_chars.gt(0), "normalized_text"].duplicated(keep=False).sum())},
]))
sns.histplot(data=thai, x="text_chars", hue="label", bins=60, element="step", stat="density", common_norm=False)
plt.title("Transcript character length (not Thai word count)"); plt.xlabel("Characters")
save_plot("05_text_lengths")


## 8. ตรวจความซ้ำ / overlap ข้าม split ก่อนใช้ฝึกหรือประเมิน

แยก 4 ความหมาย: row_id, utterance_id, normalized text, file SHA-256
utterance_id ซ้ำอาจมาจากคนละ source; text ซ้ำอาจตั้งใจให้จริง/ปลอมพูดข้อความเดียวกัน
file hash ตรงกันคือไฟล์เหมือนกันทุกไบต์ แต่ hash ต่างกันไม่ได้พิสูจน์ว่าเป็นเสียง/ผู้พูดคนละคน
**ไม่แก้ official split และไม่ลบคลิปจากผลตรวจอัตโนมัติ** หากพบ overlap ต้องสอบทานก่อน


In [ ]:
overlap_rows = []
for split_a, split_b in itertools.combinations(sorted(thai.manifest_split.unique()), 2):
    a = thai.loc[thai.manifest_split.eq(split_a)]
    b = thai.loc[thai.manifest_split.eq(split_b)]
    for column in ["row_id", "utterance_id", "normalized_text"]:
        set_a, set_b = set(values(a, column)) - {""}, set(values(b, column)) - {""}
        shared = set_a & set_b
        overlap_rows.append({"split_a": split_a, "split_b": split_b, "key": column,
            "shared_unique_values": len(shared), "clips_in_a": int(values(a, column).isin(shared).sum()),
            "clips_in_b": int(values(b, column).isin(shared).sum())})
overlap = pd.DataFrame(overlap_rows, columns=["split_a", "split_b", "key", "shared_unique_values", "clips_in_a", "clips_in_b"])
show_table("split_metadata_overlap", overlap)
within_overlap = []
for split, group in thai.groupby("manifest_split"):
    nonblank = group.loc[group.text_chars.gt(0)]
    real_text = set(nonblank.loc[nonblank.label.eq("bonafide"), "normalized_text"])
    spoof_text = set(nonblank.loc[nonblank.label.eq("spoof"), "normalized_text"])
    within_overlap.append({"split": split, "unique_nonblank_text": nonblank.normalized_text.nunique(),
                          "texts_shared_real_spoof": len(real_text & spoof_text)})
show_table("within_split_paired_text", pd.DataFrame(within_overlap))


## 9. ตรวจ header เสียงไทยทุกไฟล์ + hash (ไม่ resample ไม่แก้ไฟล์)

ตรวจ file existence/readability, duration, channels, sample rate, format, subtype, size
พาธต้องอยู่ใต้ data/processed/sea_spoof_th/audio เท่านั้น
hash อ่านไฟล์ทุกไบต์แต่ไม่ถอดรหัส waveform; เปรียบเทียบ duplicate file ทั้งภายในและข้าม split
ข้อผิดพลาดแสดงแยกและส่งออกตารางรายไฟล์ไว้ตรวจ ไม่ซ่อนเป็นค่าว่าง


In [ ]:
start = time.perf_counter()
header_tasks = [(ROOT, str(r.row_id), str(r.path), HASH_AUDIO) for r in thai.itertuples(index=False)]
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    header_records = []
    for index, result in enumerate(pool.map(inspect_audio_header, header_tasks), 1):
        header_records.append(result)
        if index % 5000 == 0:
            print(f"Header/hash progress: {index:,}/{len(header_tasks):,}", flush=True)
    headers = pd.DataFrame(header_records)
audio = thai.merge(headers, on="row_id", validate="one_to_one")
good_audio = audio.loc[audio.audio_status.eq("ok")].copy()
header_errors = audio.loc[audio.audio_status.ne("ok"), ["row_id", "path", "audio_error"]]
TABLES["audio_header_errors"] = header_errors
print(f"Header scan: {len(good_audio):,}/{len(audio):,} readable, errors={len(header_errors):,}, elapsed={time.perf_counter()-start:.1f}s")
if not len(good_audio):
    raise RuntimeError("No readable Thai audio: inspect outputs/audio_header_errors.csv.")
header_checks = pd.DataFrame([
 {"check": "unreadable/missing/path error", "clips": len(header_errors)},
 {"check": "actual sample rate not 16000", "clips": int(good_audio.actual_sample_rate.ne(16000).sum())},
 {"check": "header disagrees with metadata sample rate", "clips": int(good_audio.actual_sample_rate.ne(
     pd.to_numeric(good_audio.sampling_rate, errors="coerce")).sum())},
 {"check": "channels not mono", "clips": int(good_audio.channels.ne(1).sum())},
 {"check": "format not FLAC", "clips": int(good_audio.format.ne("FLAC").sum())},
])
show_table("audio_header_checks", header_checks)
show_table("audio_formats", grouped_counts(good_audio, ["actual_sample_rate", "channels", "format", "subtype"]))
audio_summary = good_audio.groupby(["manifest_split", "label"]).agg(
    clips=("row_id", "size"), total_seconds=("duration_s", "sum"), min_s=("duration_s", "min"),
    median_s=("duration_s", "median"), mean_s=("duration_s", "mean"), max_s=("duration_s", "max"),
    total_bytes=("file_bytes", "sum")).reset_index()
audio_summary["hours"] = audio_summary.total_seconds / 3600
audio_summary["size_GiB"] = audio_summary.total_bytes / 1024**3
show_table("audio_duration_hours", audio_summary)
show_table("duration_percentiles", good_audio.groupby(["manifest_split", "label"]).duration_s.describe(
    percentiles=[.01, .05, .25, .5, .75, .95, .99]).reset_index())
hash_overlap_rows = []
hash_groups = pd.DataFrame()
if HASH_AUDIO:
    hash_groups = good_audio.groupby("audio_sha256").agg(
        clips=("row_id", "size"), splits=("manifest_split", "nunique"), labels=("label", "nunique")).reset_index()
    hash_duplicate_summary = pd.DataFrame([
        {"check": "repeated file hash groups", "count": int(hash_groups.clips.gt(1).sum())},
        {"check": "clips in repeated file hash groups", "count": int(hash_groups.loc[hash_groups.clips.gt(1), "clips"].sum())},
        {"check": "hash groups with conflicting labels", "count": int(hash_groups.labels.gt(1).sum())},
        {"check": "hash groups crossing splits", "count": int(hash_groups.splits.gt(1).sum())}])
    show_table("file_hash_duplicate_summary", hash_duplicate_summary)
    TABLES["file_hash_duplicate_groups"] = hash_groups.loc[hash_groups.clips.gt(1)]
    repeated_hashes = set(hash_groups.loc[hash_groups.clips.gt(1), "audio_sha256"])
    TABLES["file_hash_duplicate_clips"] = good_audio.loc[good_audio.audio_sha256.isin(repeated_hashes),
        ["row_id", "path", "manifest_split", "label", "source_dataset", "source_model", "audio_sha256"]].sort_values(["audio_sha256", "manifest_split"])
    for split_a, split_b in itertools.combinations(sorted(good_audio.manifest_split.unique()), 2):
        hashes_a = set(good_audio.loc[good_audio.manifest_split.eq(split_a), "audio_sha256"])
        hashes_b = set(good_audio.loc[good_audio.manifest_split.eq(split_b), "audio_sha256"])
        hash_overlap_rows.append({"split_a": split_a, "split_b": split_b, "shared_file_hashes": len(hashes_a & hashes_b)})
    show_table("split_file_hash_overlap", pd.DataFrame(hash_overlap_rows))
else:
    display(Markdown("**File-byte duplicate check skipped:** HASH_AUDIO=False. ไม่สรุปว่าไม่มีเสียงซ้ำ."))


## 10. Distribution เสียง: ความยาว, ชั่วโมง, file size และ outliers

กราฟแยก label/split; histogram percentile zoom เป็นการแสดงผลเท่านั้น ไม่ได้ลบ long clips
IQR outlier เป็นรายการให้ตรวจ ไม่ใช่เกณฑ์ตัดเสียงออกอัตโนมัติ
เปรียบเทียบค่าบรรยายระหว่าง Dev/Test เพื่อเข้าใจข้อมูล ไม่ใช้คะแนนโมเดลหรือ Test เพื่อเลือก hyperparameters


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(data=good_audio, x="duration_s", hue="label", bins=80,
             stat="density", common_norm=False, element="step", ax=axes[0])
sns.ecdfplot(data=good_audio, x="duration_s", hue="label", ax=axes[1])
axes[0].set_title("Full clip duration distribution"); axes[1].set_title("Duration ECDF")
save_plot("06_duration_hist_ecdf")
sns.boxplot(data=good_audio, x="manifest_split", y="duration_s", hue="label", showfliers=False)
plt.title("Clip duration: boxplot (outliers hidden here only)"); plt.ylabel("Seconds")
save_plot("07_duration_boxplot")
audio_summary.pivot(index="manifest_split", columns="label", values="hours").plot.bar(stacked=True)
plt.title("Thai audio hours by split and label"); plt.ylabel("Hours")
save_plot("08_hours")
sns.scatterplot(data=good_audio, x="duration_s", y="file_bytes", hue="label", alpha=.2, s=10)
plt.title("File bytes vs duration (encoding/source effects possible)")
save_plot("09_file_size_duration")
duration_bins = [0, 1, 2, WINDOW_S, 10, 30, np.inf]
duration_labels = ["<1s", "1-2s", "2-4.0375s", "4.0375-10s", "10-30s", ">=30s"]
binned = good_audio.assign(duration_band=pd.cut(good_audio.duration_s, bins=duration_bins,
    labels=duration_labels, right=False))
show_table("duration_bands", binned.groupby(["manifest_split", "label", "duration_band"],
    observed=True).size().rename("clips").reset_index())
good_audio["duration_iqr_outlier"] = False
for _, group in good_audio.groupby(["manifest_split", "label"]):
    q1, q3 = group.duration_s.quantile([.25, .75])
    good_audio.loc[group.index, "duration_iqr_outlier"] = (
        group.duration_s.lt(q1 - 1.5 * (q3 - q1)) | group.duration_s.gt(q3 + 1.5 * (q3 - q1)))
show_table("duration_outlier_counts", good_audio.groupby(["manifest_split", "label"]).duration_iqr_outlier.agg(
    ["size", "sum"]).reset_index().rename(columns={"size": "clips", "sum": "IQR_flagged_clips"}))
TABLES["duration_outlier_details"] = good_audio.loc[good_audio.duration_iqr_outlier,
    ["row_id", "path", "manifest_split", "label", "duration_s"]]
good_audio["audio_source"] = np.where(good_audio.label.eq("bonafide"),
    category_values(good_audio, "source_dataset"), category_values(good_audio, "source_model"))
show_table("duration_by_audio_source", good_audio.groupby(["manifest_split", "label", "audio_source"]).duration_s.agg(
    ["count", "median", "mean", "min", "max"]).reset_index())


## 11. ผลของ input window 4.0375 วินาที

นี่คือ **วิธีเตรียม input ใน baseline** ไม่ใช่ความยาวสูงสุดของไฟล์ที่รับ
คำนวณว่าไฟล์ใดจะถูก trim/repeat-pad และสัดส่วนเสียงต้นฉบับที่ได้ใช้
retained_original_fraction ของไฟล์ยาว = 4.0375 / duration
padded_fraction_of_input ของไฟล์สั้น = ส่วนที่ต้องเติม / 4.0375
ใช้ duration ประเมิน หาก sample rate ต่างจาก 16 kHz อาจต่างจากผล resampler จริงเล็กน้อย
ไม่ได้ปรับ AASIST/RawNet2 หรือเลือกระยะ window จาก Test ใน notebook นี้


In [ ]:
window_records = [input_window_summary(int(r.frames), int(r.actual_sample_rate),
    TARGET_SAMPLES, TARGET_SAMPLE_RATE) for r in good_audio.itertuples(index=False)]
window_frame = pd.DataFrame(window_records, index=good_audio.index)
good_audio = pd.concat([good_audio, window_frame], axis=1)
window_counts = grouped_counts(good_audio, ["manifest_split", "label", "window_action"])
window_counts["pct_within_split_label"] = window_counts.clips / window_counts.groupby(
    ["manifest_split", "label"]).clips.transform("sum") * 100
show_table("baseline_window_actions", window_counts)
show_table("baseline_window_coverage", good_audio.groupby(["manifest_split", "label"]).agg(
    clips=("row_id", "size"), mean_retained_original_fraction=("retained_original_fraction", "mean"),
    median_retained_original_fraction=("retained_original_fraction", "median"),
    mean_padded_fraction_of_input=("padded_fraction_of_input", "mean")).reset_index())
pd.crosstab(good_audio.manifest_split, good_audio.window_action, normalize="index").plot.bar(stacked=True)
plt.title("Baseline input window: crop / repeat padding"); plt.ylabel("Fraction of clips")
save_plot("10_window_actions")


## 12. สำรวจ waveform quality: sample แบบ reproducible หรือทั้งชุด

ค่าเริ่มต้นสุ่มไม่เกิน 60 คลิปต่อ split × label จึงไม่ใช่สัดส่วนตามประชากรทั้งชุด
ทุกค่าต่อไปนี้ถอดรหัส **ทั้งคลิปที่เลือก** แบบ block เพื่อลด RAM และไม่ normalize/resample:
- RMS/dBFS: ความแรงสัญญาณดิจิทัล ไม่ใช่ loudness perceptual หรือระดับเสียงจริงหน่วย dB SPL
- near_zero_fraction: samples ที่ |amplitude| < 1e-4 ไม่ใช่ voice activity detector
- near_full_scale_fraction: samples ที่ channel ใด channel หนึ่ง |amplitude| >= .999 เป็น warning ไม่ใช่ยืนยัน clipping
- DC offset, zero crossings, non-finite / all-zero
ห้ามใช้ค่าเหล่านี้เป็นกฎตัดสินเสียงปลอม และไม่ประมาณ SNR หากไม่มีเสียงสะอาดอ้างอิง


In [ ]:
if SIGNAL_SCOPE == "all":
    selected = good_audio.copy()
else:
    selected = pd.concat([group.sort_values("row_id").sample(
        n=min(SAMPLES_PER_SPLIT_LABEL, len(group)), random_state=SEED)
        for _, group in good_audio.groupby(["manifest_split", "label"])], ignore_index=True)
coverage = grouped_counts(selected, ["manifest_split", "label"]).rename(columns={"clips": "decoded_clips"})
coverage = coverage.merge(grouped_counts(good_audio, ["manifest_split", "label"]).rename(
    columns={"clips": "available_clips"}), on=["manifest_split", "label"], validate="one_to_one")
coverage["coverage_pct"] = coverage.decoded_clips / coverage.available_clips * 100
show_table("signal_scan_coverage", coverage)
signal_tasks = [(ROOT, str(r.row_id), str(r.path), NEAR_ZERO_THRESHOLD) for r in selected.itertuples(index=False)]
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    decoded_records = []
    for index, result in enumerate(pool.map(inspect_signal, signal_tasks), 1):
        decoded_records.append(result)
        if index % 5000 == 0:
            print(f"Waveform progress: {index:,}/{len(signal_tasks):,}", flush=True)
    signal_rows = pd.DataFrame(decoded_records)
signals = selected.merge(signal_rows, on="row_id", validate="one_to_one")
signal_good = signals.loc[signals.signal_status.eq("ok")].copy()
TABLES["signal_errors"] = signals.loc[signals.signal_status.ne("ok"), ["row_id", "path", "signal_error"]]
metrics = ["mono_peak", "rms_dbfs", "near_zero_fraction", "near_full_scale_fraction", "dc_offset", "zero_crossing_rate"]
show_table("signal_metric_summary", signal_good.groupby(["manifest_split", "label"])[metrics].agg(["count", "mean", "median", "min", "max"]).reset_index())
show_table("signal_warning_counts", pd.DataFrame([
 {"check": "decode errors", "clips": int(signals.signal_status.ne("ok").sum())},
 {"check": "all-zero after mono averaging", "clips": int(signal_good.all_zero_mono.sum())},
 {"check": "any near-full-scale sample", "clips": int(signal_good.near_full_scale_fraction.gt(0).sum())},
 {"check": "near-zero sample fraction > 90% (heuristic only)", "clips": int(signal_good.near_zero_fraction.gt(.9).sum())}]))
if len(signal_good):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, metric in zip(axes, ["rms_dbfs", "near_zero_fraction", "zero_crossing_rate"]):
        sns.boxplot(data=signal_good, x="label", y=metric, ax=ax)
        ax.set_title(metric)
    plt.suptitle(f"Signal metrics: {SIGNAL_SCOPE}, n={len(signal_good)}")
    save_plot("11_signal_metrics")
display(Markdown(f"**การตีความ:** สถิติสัญญาณนี้ครอบคลุม {len(selected):,} จาก {len(good_audio):,} คลิป "
                 f"ในโหมด {SIGNAL_SCOPE}; ความต่างรายกลุ่มยังไม่พิสูจน์ว่าเกิดจากการสังเคราะห์เสียง."))


## 13. ตัวอย่าง waveform / spectrum / spectrogram และฟังเสียง (optional)

เลือก 1 คลิปต่อ label ใกล้ median duration ของแต่ละ label เพื่อสาธิต ไม่ใช่ตัวแทนทั้งชุด
กราฟใช้ช่วงเริ่มต้นไม่เกิน 8 วินาที ไม่ใช่ waveform ทั้งไฟล์ยาว
Welch spectrum และ spectrogram แสดงลักษณะสัญญาณ ไม่ยืนยันความจริง/ปลอมด้วยสายตา
ฟังเสียงได้เมื่อ PLAY_AUDIO=True เฉพาะในเครื่องที่มีสิทธิ์; การเปิดนี้จะฝังเสียงใน output notebook


In [ ]:
example_rows = []
for label, group in good_audio.groupby("label"):
    median = group.duration_s.median()
    ranked = group.assign(distance_from_median=(group.duration_s - median).abs()).sort_values(
        ["distance_from_median", "row_id"])
    example_rows.append(ranked.iloc[0])
for example in example_rows:
    path = resolve_audio_path(ROOT, str(example["path"]))
    with sf.SoundFile(path) as handle:
        sr = handle.samplerate
        waveform = handle.read(frames=min(handle.frames, sr * 8), dtype="float64", always_2d=True).mean(axis=1)
    t = np.arange(len(waveform)) / sr
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(t, waveform, linewidth=.5)
    axes[0].set(xlabel="Seconds", ylabel="Amplitude", title="Waveform (first <=8s)")
    frequencies, power = scipy_signal.welch(waveform, fs=sr, nperseg=min(1024, len(waveform)))
    axes[1].plot(frequencies, 10 * np.log10(np.maximum(power, 1e-15)))
    axes[1].set(xlabel="Hz", ylabel="PSD (dB/Hz)", title="Welch power spectrum")
    axes[2].specgram(waveform, NFFT=min(512, len(waveform)), Fs=sr,
        noverlap=min(384, max(min(512, len(waveform)) - 1, 0)), cmap="magma")
    axes[2].set(xlabel="Seconds", ylabel="Hz", title="Spectrogram")
    plt.suptitle(f"Illustrative {example['label']} clip — not a detection rule")
    save_plot("12_example_" + str(example["label"]))
    if SHOW_RAW_EXAMPLES:
        display({"row_id": example["row_id"], "text": example["text"], "source": example["audio_source"]})
    if PLAY_AUDIO:
        display(Audio(waveform, rate=sr, normalize=False))


## 14. สรุปที่ใช้คุยอาจารย์ได้ + export เพื่อทำซ้ำ

สรุปจากข้อมูลที่รันจริง ไม่ hardcode จำนวนคลิปจากรายงานรอบก่อน
บันทึก scope / versions / seed / revision / manifest hash และจำนวนข้อผิดพลาด
ผลรายแถว/กราฟ/เสียง-derived data อยู่ใน outputs/ ซึ่งไม่เข้า Git
Notebook source มี output ว่างเพื่อแชร์โค้ดได้; executed notebook และผลละเอียดไม่ควรเผยแพร่โดยไม่ตรวจสิทธิ์

คำถามที่ยังต้องสอบทานกับผู้สร้าง/อาจารย์:
- mapping อายุ เพศ และรหัสผู้พูดของเสียงจริงมีที่ต้นทางหรือไม่?
- official split แยกผู้พูด / source utterance อย่างไร?
- ถิ่นหมายถึงที่อยู่ ภูมิลำเนา หรือสำเนียง?
- ยอมรับชุดทดสอบเสริมประชากรหรือข้อมูลประชากรต้องครบในชุดหลัก?
- ไม่อ้างว่าไม่มี duplicate speaker เพียงเพราะ row_id/file hash ไม่ซ้ำ


In [ ]:
summary = {
 "dataset": "Jack-ppkdczgx/SEA-Spoof",
 "source_card": "https://huggingface.co/datasets/Jack-ppkdczgx/SEA-Spoof",
 "started_at_utc": STARTED_AT, "finished_at_utc": datetime.now(timezone.utc).isoformat(),
 "scope": {"parquet_splits": available_splits, "shards": len(shards),
     "metadata_rows_all_languages": len(source), "thai_source_rows": len(source_thai),
     "thai_manifest_rows": len(thai), "thai_audio_readable": len(good_audio),
     "audio_header_errors": len(header_errors), "signal_scope": SIGNAL_SCOPE,
     "signal_selected_clips": len(selected), "signal_decoded_clips": len(signal_good),
     "file_sha256_checked": HASH_AUDIO, "speaker_disjoint_verified": False},
 "settings": {"seed": SEED, "workers": WORKERS, "samples_per_split_label": SAMPLES_PER_SPLIT_LABEL,
     "near_zero_threshold": NEAR_ZERO_THRESHOLD, "target_samples": TARGET_SAMPLES,
     "target_sample_rate": TARGET_SAMPLE_RATE, "window_s": WINDOW_S,
     "play_audio": PLAY_AUDIO, "show_raw_examples": SHOW_RAW_EXAMPLES},
 "versions": versions, "python": platform.python_version(), "python_executable": sys.executable,
 "parquet_inventory": inventory.to_dict(orient="records"),
 "manifest_inventory": manifest_inventory,
 "thai_label_counts": label_counts.to_dict(orient="records"),
 "audio_summary": audio_summary.to_dict(orient="records"),
 "demographic_fields": demographics.to_dict(orient="records"),
 "file_hash_duplicate_summary": hash_duplicate_summary.to_dict(orient="records") if HASH_AUDIO else None,
 "cross_split_identical_file_hash_groups": int(hash_groups.splits.gt(1).sum()) if HASH_AUDIO else None,
 "limitations": [
     "Only local shards; missing Train is not summarized.",
     "No model inference, training, threshold selection, or split changes.",
     "No demographic prediction; voice IDs are not verified person IDs.",
     "SHA-256 checks byte-identical files, not re-encodings or speaker identity.",
     "Signal sample is split-label stratified; not a population-weighted estimate.",
     "Transcript overlap alone does not prove leaked audio or speaker overlap.",
     "Source audio bytes are not read for full source/manifest byte equality.",
     "No perceptual quality, VAD, SNR, or human listening validation in default mode."
 ]}
TABLES["thai_audio_headers"] = audio.drop(columns=["text", "normalized_text"], errors="ignore")
TABLES["signal_clip_metrics"] = signals.drop(columns=["text", "normalized_text"], errors="ignore")
TABLES["raw_example_selection"] = pd.DataFrame([{"row_id": r["row_id"], "path": r["path"],
    "label": r["label"], "duration_s": r["duration_s"]} for r in example_rows])
for name, table in TABLES.items():
    table.to_csv(OUTPUT / (name + ".csv"), index=False, encoding="utf-8-sig")
(OUTPUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
(OUTPUT / "source_schemas.json").write_text(json.dumps(schemas, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
display(Markdown(
    f"### ข้อค้นพบจากการรันรอบนี้\n"
    f"- Parquet ในเครื่อง: **{len(source):,} แถว / {len(shards)} shards**, split={available_splits}\n"
    f"- ไทยใน manifest: **{len(thai):,} คลิป**; header อ่านได้ **{len(good_audio):,} คลิป**\n"
    f"- ชั่วโมงเสียงไทยที่อ่านได้: **{good_audio.duration_s.sum()/3600:.2f} ชั่วโมง**\n"
    f"- สัญญาณที่ถอดรหัส: **{len(signal_good):,}/{len(good_audio):,} คลิป**, mode={SIGNAL_SCOPE}\n"
    f"- age/gender/region fields: **{demographics.available_columns.tolist()}**\n"
    f"- สิ่งที่ยังยืนยันไม่ได้: จำนวนบุคคลไม่ซ้ำ, speaker-disjoint splits, ประชากรครบทุกวัย/ถิ่น\n"
    f"- ไฟล์ hash ตรงกันข้าม split: **{summary['cross_split_identical_file_hash_groups']} กลุ่ม** (None=ไม่ได้ตรวจ); ต้องสอบทาน ไม่เปลี่ยน split อัตโนมัติ\n"
    f"- ยังไม่ใช่ผล baseline AASIST/RawNet2 และยังไม่ใช่หลักฐานความแม่นยำโมเดล\n"
    f"- ส่งออก **{len(TABLES)} ตาราง** พร้อมกราฟและ summary.json ใต้ data/exploration/sea_spoof/outputs/"
))
print("EDA complete. Source Parquet, manifests and FLAC files were not modified.")
